# Snowflake churn: selected paper figures

Run the cells in order. Each plot is displayed and saved as a PDF in `notebooks/paper_ready_figures/`. The enumeration figure can use `restricted.csv` and `unrestricted.csv` directly; countries are looked up by ASN in `ipinfo_asn.csv`. Other figures may still require enriched measurement CSVs. Set `IPINFO_ASN_CSV` if the metadata lives elsewhere. Measurements after 2025-06-24 are excluded.


In [ ]:
import csv
import os
import sys
from collections import Counter, defaultdict
from datetime import date, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

def find_repo_root():
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / 'analysis' / 'snowflake_churn.py').is_file():
            return candidate.resolve()
    raise FileNotFoundError('Run this notebook from the repository root or notebooks directory')

REPO_ROOT = find_repo_root()
FIGURE_OUTPUT_DIR = REPO_ROOT / 'notebooks' / 'paper_ready_figures'
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IPINFO_ASN_CSV = Path(os.environ.get(
    'IPINFO_ASN_CSV', str(REPO_ROOT / 'ipinfo_asn.csv')
))
ANALYSIS_END_DATE = date(2025, 6, 24)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analysis.snowflake_churn import (
    attach_asn_metadata, combine_pools, compute_daily_churn,
    daily_entity_sets, load_dataset, load_ipinfo_asn_metadata,
)

def save_paper_figure(fig, filename):
    fig.savefig(FIGURE_OUTPUT_DIR / filename, format='pdf', bbox_inches='tight')
    plt.show()
    plt.close(fig)

def format_axes(ax):
    for spine in ('top', 'right'):
        ax.spines[spine].set_color('gray')
        ax.spines[spine].set_linewidth(0.5)
    for spine in ('left', 'bottom'):
        ax.spines[spine].set_color('gray')
        ax.spines[spine].set_linewidth(0.5)
    ax.xaxis.set_ticks_position('bottom')
    ax.yaxis.set_ticks_position('left')
    for axis in (ax.xaxis, ax.yaxis):
        axis.set_tick_params(direction='in', color='gray')
    return ax

def get_asn_type(row):
    metadata = row.get('asn_metadata')
    return (metadata or {}).get('type', '').strip() or 'unknown'

def style_paper_axes(ax):
    ax.grid(True, linestyle=':', linewidth=0.8, color='black', alpha=0.8)
    ax.tick_params(axis='both', labelsize=18, direction='in')
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)

plt.style.use('seaborn-v0_8-whitegrid')
print('Figure output:', FIGURE_OUTPUT_DIR)


## Load measurements and figure inputs


In [ ]:
enriched_inputs_available = all(
    (REPO_ROOT / f'{pool}_with_country_with_rank.csv').is_file()
    for pool in ('restricted', 'unrestricted')
)
restricted_rows_all = load_dataset(REPO_ROOT, 'restricted', enriched=enriched_inputs_available)
unrestricted_rows_all = load_dataset(REPO_ROOT, 'unrestricted', enriched=enriched_inputs_available)
restricted_rows = [row for row in restricted_rows_all if row['date'] <= ANALYSIS_END_DATE]
unrestricted_rows = [row for row in unrestricted_rows_all if row['date'] <= ANALYSIS_END_DATE]

def load_timestamped_raw_rows(pool):
    path = REPO_ROOT / {
        'restricted': 'restricted.csv',
        'unrestricted': 'unrestricted.csv',
    }[pool]
    rows = []
    with path.open(newline='') as handle:
        for row in csv.reader(handle):
            if len(row) < 2:
                continue
            timestamp = datetime.fromisoformat(row[0].replace('Z', '+00:00'))
            if timestamp.date() <= ANALYSIS_END_DATE:
                rows.append((timestamp, row[1]))
    return rows

timestamped_raw = {pool: load_timestamped_raw_rows(pool) for pool in ('restricted', 'unrestricted')}
ipinfo_metadata = load_ipinfo_asn_metadata(IPINFO_ASN_CSV)
for row in restricted_rows + unrestricted_rows:
    metadata = ipinfo_metadata.get(row.get('asn')) or {}
    row['country'] = metadata.get('country', '').strip() or None
all_rows = combine_pools({'restricted': restricted_rows, 'unrestricted': unrestricted_rows})

restricted_ip_daily = daily_entity_sets(restricted_rows, 'ip_hash')
unrestricted_ip_daily = daily_entity_sets(unrestricted_rows, 'ip_hash')
combined_ip_daily = daily_entity_sets(all_rows, 'ip_hash')
restricted_asn_daily = daily_entity_sets(restricted_rows, 'asn')
unrestricted_asn_daily = daily_entity_sets(unrestricted_rows, 'asn')
combined_asn_daily = daily_entity_sets(all_rows, 'asn')
combined_ip_churn = compute_daily_churn(combined_ip_daily)

observed_asns = {row['asn'] for row in all_rows if row['asn'] is not None}
unique_asns_by_type = Counter(
    (ipinfo_metadata.get(asn, {}) or {}).get('type', '').strip() or 'unknown'
    for asn in observed_asns
)

IRAN_CONTEXT_WINDOWS = [
    {'label': 'Iran traffic ramp-up 1', 'start': date(2025, 6, 12), 'end': date(2025, 6, 18), 'color': '#f6c85f', 'alpha': 0.18},
    {'label': 'Iran shutdown', 'start': date(2025, 6, 18), 'end': date(2025, 6, 21), 'color': '#e45756', 'alpha': 0.16},
    {'label': 'Iran traffic ramp-up 2', 'start': date(2025, 6, 21), 'end': date(2025, 6, 24), 'color': '#72b7b3', 'alpha': 0.16},
]
print(f'Loaded {len(restricted_rows):,} restricted and {len(unrestricted_rows):,} unrestricted measurements')


## 1. Enumeration: proxies, ASNs, countries


In [ ]:
def rolling_average(values, window=5):
    if len(values) < window:
        return values
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode='same')

def first_seen_daily_counts(daily_sets):
    dates = sorted(daily_sets)
    seen = set()
    novel_counts = []
    cumulative_counts = []
    final_total = len(set().union(*daily_sets.values()))
    for day in dates:
        current = daily_sets[day]
        novel_today = current - seen
        seen |= current
        novel_counts.append(len(novel_today))
        cumulative_counts.append(len(seen))
    return dates, novel_counts, cumulative_counts, final_total

restricted_country_daily = daily_entity_sets(restricted_rows, 'country')
unrestricted_country_daily = daily_entity_sets(unrestricted_rows, 'country')
combined_country_daily = daily_entity_sets(all_rows, 'country')
combined_successes_by_day = defaultdict(int)
for pool in ('restricted', 'unrestricted'):
    for timestamp, _ in timestamped_raw[pool]:
        combined_successes_by_day[timestamp.date()] += 1

paper_ready_entity_sets = {
    'IP hashes': {'restricted': restricted_ip_daily, 'unrestricted': unrestricted_ip_daily, 'combined': combined_ip_daily},
    'ASNs': {'restricted': restricted_asn_daily, 'unrestricted': unrestricted_asn_daily, 'combined': combined_asn_daily},
    'Countries': {'restricted': restricted_country_daily, 'unrestricted': unrestricted_country_daily, 'combined': combined_country_daily},
}
paper_ready_series = {}
for label, pool_sets in paper_ready_entity_sets.items():
    combined_dates, novel_counts, combined_cumulative_counts, final_total = first_seen_daily_counts(pool_sets['combined'])
    novelty_yield = [novel_count / combined_successes_by_day[day] * 100 for day, novel_count in zip(combined_dates, novel_counts)]
    restricted_dates, _, restricted_cumulative_counts, _ = first_seen_daily_counts(pool_sets['restricted'])
    unrestricted_dates, _, unrestricted_cumulative_counts, _ = first_seen_daily_counts(pool_sets['unrestricted'])
    paper_ready_series[label] = {
        'dates': combined_dates,
        'restricted_dates': restricted_dates,
        'unrestricted_dates': unrestricted_dates,
        'restricted_cumulative_counts': restricted_cumulative_counts,
        'unrestricted_cumulative_counts': unrestricted_cumulative_counts,
        'novelty_yield': novelty_yield,
    }


In [ ]:
panel_styles = {
    'IP hashes': {'restricted_marker': 'o', 'unrestricted_marker': 's', 'yield_marker': 'D'},
    'ASNs': {'restricted_marker': '^', 'unrestricted_marker': 'v', 'yield_marker': 'P'},
    'Countries': {'restricted_marker': '<', 'unrestricted_marker': '>', 'yield_marker': 'X'},
}

fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharex=False)
for ax, label in zip(axes, ['IP hashes', 'ASNs', 'Countries']):
    series = paper_ready_series[label]
    dates = series['dates']
    ax.plot(
        series['restricted_dates'],
        series['restricted_cumulative_counts'],
        color='tab:blue',
        linewidth=2.4,
        marker=panel_styles[label]['restricted_marker'],
        markersize=3.5,
        label='Restricted Proxies',
    )
    ax.plot(
        series['unrestricted_dates'],
        series['unrestricted_cumulative_counts'],
        color='tab:green',
        linewidth=2.4,
        marker=panel_styles[label]['unrestricted_marker'],
        markersize=3.5,
        label='Unrestricted Proxies',
    )
    left_axis_labels = {
        'IP hashes': 'Unique #Proxies (cum.)',
        'ASNs': 'Unique #ASNs (cum.)',
        'Countries': 'Unique #Countries (cum.)',
    }
    ax.set_ylabel(left_axis_labels[label], color='tab:blue', fontsize=24)
    ax.tick_params(axis='y', labelcolor='tab:blue')
    style_paper_axes(ax)
    for window in IRAN_CONTEXT_WINDOWS:
        ax.axvspan(window['start'], window['end'], color=window['color'], alpha=window['alpha'])

    ax2 = ax.twinx()
    ax2.plot(
        dates,
        series['novelty_yield'],
        color='tab:red',
        linewidth=2.0,
        linestyle='--',
        marker=panel_styles[label]['yield_marker'],
        markersize=3.0,
        label='Novel Yield / Response',
    )
    ax2.set_ylabel('Novel / Response (%)', color='tab:red', fontsize=24)
    ax2.tick_params(axis='y', colors='tab:red', labelsize=20, direction='in')
    ax2.yaxis.set_label_position('right')
    ax2.yaxis.tick_right()
    ax2.spines['right'].set_color('tab:red')
    ax2.spines['right'].set_linewidth(0.8)
    ax2.spines['left'].set_visible(False)
    ax.right_ax = ax2

for ax in axes:
    ax.tick_params(axis='x', rotation=35)
    ax.tick_params(axis='both', labelsize=22)

    format_axes(ax)

handles = [
    axes[0].lines[0],
    axes[0].lines[1],
    axes[0].right_ax.lines[0],
    Patch(facecolor=IRAN_CONTEXT_WINDOWS[0]['color'], alpha=IRAN_CONTEXT_WINDOWS[0]['alpha'], label=IRAN_CONTEXT_WINDOWS[0]['label']),
    Patch(facecolor=IRAN_CONTEXT_WINDOWS[1]['color'], alpha=IRAN_CONTEXT_WINDOWS[1]['alpha'], label=IRAN_CONTEXT_WINDOWS[1]['label']),
    Patch(facecolor=IRAN_CONTEXT_WINDOWS[2]['color'], alpha=IRAN_CONTEXT_WINDOWS[2]['alpha'], label=IRAN_CONTEXT_WINDOWS[2]['label']),
]
labels = [
    'Restricted Proxies',
    'Unrestricted Proxies',
    'Novel Yield / Response',
    IRAN_CONTEXT_WINDOWS[0]['label'],
    IRAN_CONTEXT_WINDOWS[1]['label'],
    IRAN_CONTEXT_WINDOWS[2]['label'],
]

fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.12), ncol=3, fontsize=23, frameon=True)
plt.tight_layout(rect=(0, 0, 1, 0.9))
format_axes(ax)
save_paper_figure(fig, 'enumeration-proxies-asns-countries_figure.pdf')
print(FIGURE_OUTPUT_DIR / 'enumeration-proxies-asns-countries_figure.pdf')

## 2. Active and retained proxies


In [ ]:
event_styles = {
    'IP hashes': {'marker': 'o'},
}

panel_colors = {
    'retained': '#4e79a7',
    'new': '#f28e2b',
    'total': '#222222',
}

fig, ax = plt.subplots(figsize=(8.2, 4.3))

label = 'IP hashes'
metrics = combined_ip_churn

dates = [entry['date'] for entry in metrics]
retained = [0 if entry['retained_count'] is None else entry['retained_count'] for entry in metrics]
totals = [entry['current_count'] for entry in metrics]

ax.fill_between(
    dates, 0, retained,
    color=panel_colors['retained'],
    edgecolor='#2f2f2f',
    linewidth=0.8,
    hatch='///',
    alpha=0.68,
    label='Retained From Previous Day',
)
ax.fill_between(
    dates, retained, totals,
    color=panel_colors['new'],
    edgecolor='#2f2f2f',
    linewidth=0.8,
    hatch='...',
    alpha=0.68,
    label='New Relative To Previous Day',
)
ax.plot(
    dates, totals,
    color=panel_colors['total'],
    linewidth=1.8,
    marker=event_styles[label]['marker'],
    markersize=3.5,
    label='Observed Total',
)

for window in IRAN_CONTEXT_WINDOWS:
    ax.axvspan(window['start'], window['end'], color=window['color'], alpha=window['alpha'])

ax.set_ylabel('Observed Proxies', fontsize=24)
style_paper_axes(ax)
ax.tick_params(axis='x', rotation=35)
format_axes(ax)

handles = [
    Patch(facecolor=panel_colors['retained'], edgecolor='#2f2f2f', hatch='///', alpha=0.68, label='Retained From Previous Day'),
    Patch(facecolor=panel_colors['new'], edgecolor='#2f2f2f', hatch='...', alpha=0.68, label='New Relative To Previous Day'),
    ax.lines[0],
    Patch(facecolor=IRAN_CONTEXT_WINDOWS[0]['color'], alpha=IRAN_CONTEXT_WINDOWS[0]['alpha'], label=IRAN_CONTEXT_WINDOWS[0]['label']),
    Patch(facecolor=IRAN_CONTEXT_WINDOWS[1]['color'], alpha=IRAN_CONTEXT_WINDOWS[1]['alpha'], label=IRAN_CONTEXT_WINDOWS[1]['label']),
    Patch(facecolor=IRAN_CONTEXT_WINDOWS[2]['color'], alpha=IRAN_CONTEXT_WINDOWS[2]['alpha'], label=IRAN_CONTEXT_WINDOWS[2]['label']),
]

labels = [
    'Observed in Previous Day',
    'New From Previous Day',
    'Observed Total',
    IRAN_CONTEXT_WINDOWS[0]['label'],
    IRAN_CONTEXT_WINDOWS[1]['label'],
    IRAN_CONTEXT_WINDOWS[2]['label'],
]

fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.2), ncol=2, fontsize=20, frameon=True)
plt.tight_layout(rect=(0, 0, 1, 0.88))
save_paper_figure(fig, 'active-retained-proxies.pdf')
print(FIGURE_OUTPUT_DIR / 'active-retained-proxies.pdf')


## 3. ASN blocking and domain collateral


In [ ]:
BLOCKING_PERCENTAGES = list(range(0, 101))

#These are good enough for the figure? 
DOMAIN_THRESHOLDS = [100, 10_000, 1_000_000]

#From Linden's code. 
OLD_FIGURE_ANNOTATION_ASNS = ['20440', '16509', '13335', '15169']

#TODO: What to do about not found ASNs? Currently we are just ignoring those proxies wrt blocking specific ASNs, which I believe is correct. 
def _normalize_blocking_asn(value):
    if value is None:
        return None
    cleaned = str(value).strip()
    if not cleaned or cleaned.lower() in {'asnnotfound', 'asn not found', 'none', 'nan', 'unknown'}:
        return None
    if cleaned.upper().startswith('AS'):
        cleaned = cleaned[2:].strip()
    cleaned = cleaned.split(',')[0].strip()
    return cleaned if cleaned.isdigit() else None


def _parse_optional_int(value):
    if value is None:
        return None
    cleaned = str(value).strip()
    if not cleaned or cleaned in {'None', 'nan'}:
        return None
    try:
        return int(float(cleaned))
    except ValueError:
        return None

def _build_pool_asn_stats(rows):
    stats = {}
    for row in rows:
        asn = row.get('asn')
        ip_hash = row.get('ip_hash')
        if asn is None or not ip_hash:
            continue
        entry = stats.setdefault(asn, {'ips': set(), 'obs_count': 0})
        entry['ips'].add(ip_hash)
        entry['obs_count'] += 1
    return stats


def _sort_pool_asns(asn_stats, strategy):
    if strategy != 'frequency':
        raise ValueError(f'Unknown blocking strategy: {strategy}')
    return [
        asn for asn, _ in sorted(
            asn_stats.items(),
            key=lambda item: (-item[1]['obs_count'], -len(item[1]['ips']), item[0]),
        )
    ]


def _curve_count_from_percentage(total_count, percentage):
    if total_count == 0 or percentage <= 0:
        return 0
    return min(total_count, max(1, (total_count * percentage + 99) // 100))


def _blocking_coverage_curve(rows, strategy, percentages=BLOCKING_PERCENTAGES):
    asn_stats = _build_pool_asn_stats(rows)
    sorted_asns = _sort_pool_asns(asn_stats, strategy)
    total_unique_ips = len({row['ip_hash'] for row in rows if row.get('ip_hash')})
    cumulative_coverages = [0.0]
    blocked_ips = set()
    for asn in sorted_asns:
        blocked_ips |= asn_stats[asn]['ips']
        cumulative_coverages.append(0.0 if total_unique_ips == 0 else len(blocked_ips) / total_unique_ips * 100)
    total_asns = len(sorted_asns)
    curve = [cumulative_coverages[_curve_count_from_percentage(total_asns, pct)] for pct in percentages]
    return {
        'percentages': percentages,
        'curve': curve,
        'total_asns': total_asns,
        'total_ips': total_unique_ips,
        'sorted_asns': sorted_asns,
    }


def _load_top_domain_rows(root_path):
    domain_rows = []
    with (root_path / 'Top-1m_with_ips_with_asn.csv').open(newline='') as handle:
        reader = csv.reader(handle)
        for row in reader:
            if len(row) < 4:
                continue
            rank = _parse_optional_int(row[0])
            domain = str(row[1]).strip().lower()
            asns = []
            for part in str(row[3]).split(','):
                normalized = _normalize_blocking_asn(part)
                if normalized is not None:
                    asns.append(normalized)
            unique_asns = sorted(set(asns))
            if rank is None or not domain or not unique_asns:
                continue
            domain_rows.append({'rank': rank, 'domain': domain, 'asns': unique_asns})
    return domain_rows


def _domain_collateral_curve(sorted_asns, domain_rows, rank_threshold, percentages=BLOCKING_PERCENTAGES):
    filtered_rows = [row for row in domain_rows if row['rank'] <= rank_threshold]
    total_domains = len(filtered_rows)
    asn_to_domains = defaultdict(set)
    for domain_index, row in enumerate(filtered_rows):
        for asn in row['asns']:
            asn_to_domains[asn].add(domain_index)
    blocked_domains = set()
    cumulative_coverages = [0.0]
    cumulative_blocked_sets = [set()]
    for asn in sorted_asns:
        blocked_domains |= asn_to_domains.get(asn, set())
        cumulative_coverages.append(0.0 if total_domains == 0 else len(blocked_domains) / total_domains * 100)
        cumulative_blocked_sets.append(set(blocked_domains))
    total_asns = len(sorted_asns)
    curve = [cumulative_coverages[_curve_count_from_percentage(total_asns, pct)] for pct in percentages]
    return {
        'percentages': percentages,
        'curve': curve,
        'total_domains': total_domains,
        'filtered_rows': filtered_rows,
        'blocked_sets_by_percentage': {
            pct: cumulative_blocked_sets[_curve_count_from_percentage(total_asns, pct)]
            for pct in percentages
        },
    }


def _old_figure_annotations(sorted_asns, collateral_result, target_asns=OLD_FIGURE_ANNOTATION_ASNS):
    filtered_rows = collateral_result['filtered_rows']
    total_asns = len(sorted_asns)
    asn_to_domain_rows = defaultdict(list)
    for row in filtered_rows:
        for asn in row['asns']:
            asn_to_domain_rows[asn].append(row)
    annotations = []
    for asn in target_asns:
        if asn not in sorted_asns:
            continue
        index = sorted_asns.index(asn) + 1
        pct = min(100, max(1, (index * 100 + total_asns - 1) // total_asns))
        domains = sorted(asn_to_domain_rows.get(asn, []), key=lambda row: row['rank'])[:2]
        if not domains:
            continue
        curve = collateral_result['curve']
        previous_pct = max(0, pct - 1)
        rise = curve[pct] - curve[previous_pct]
        annotations.append({
            'asn': asn,
            'pct': pct,
            'rise': rise,
            'domains': domains,
        })
    return annotations


blocking_strategy_meta = {
    'frequency': {'label': 'Frequency (L->S)', 'color': '#d62728', 'marker': '^'},
}

blocking_pool_rows = {
    'Restricted proxies': restricted_rows,
    'Unrestricted proxies': unrestricted_rows,
}

blocking_coverage_results = {
    pool_label: {
        strategy: _blocking_coverage_curve(rows, strategy)
        for strategy in blocking_strategy_meta
    }
    for pool_label, rows in blocking_pool_rows.items()
}

top_domain_rows = _load_top_domain_rows(REPO_ROOT)
blocking_collateral_results = {}
for pool_label, rows in blocking_pool_rows.items():
    sorted_by_frequency = _blocking_coverage_curve(rows, 'frequency')['sorted_asns']
    blocking_collateral_results[pool_label] = {
        threshold: _domain_collateral_curve(sorted_by_frequency, top_domain_rows, threshold)
        for threshold in DOMAIN_THRESHOLDS
    }

old_figure_annotations = {
    pool_label: _old_figure_annotations(
        _blocking_coverage_curve(blocking_pool_rows[pool_label], 'frequency')['sorted_asns'],
        pool_data[100],
    )
    for pool_label, pool_data in blocking_collateral_results.items()
}

print('Blocking coverage baselines:')
for pool_label, pool_data in blocking_coverage_results.items():
    print(pool_label)
    for strategy, result in pool_data.items():
        print(f"  {blocking_strategy_meta[strategy]['label']}: {result['total_asns']:,} ASNs, {result['total_ips']:,} unique IP hashes")


In [ ]:
pool_line_meta = {
    'Restricted proxies': {'suffix': 'Res.', 'alpha': 1.0, 'linestyle': '-', 'markerfacecolor': 'white'},
    'Unrestricted proxies': {'suffix': 'Unres.', 'alpha': 0.45, 'linestyle': '--', 'markerfacecolor': None},
}


In [ ]:
domain_threshold_meta_dual = {
    100: {'label': 'Top 100 Domains', 'color': 'tab:blue', 'marker': 'o'},
    10_000: {'label': 'Top 10K Domains', 'color': 'tab:orange', 'marker': 's'},
    1_000_000: {'label': 'Top 1M Domains', 'color': 'tab:green', 'marker': '^'},
}

proxy_overlay_meta = {
    'label': 'Snowflakes Blocked',
    'color': '#c62828',
    'linestyle': '-',
    'marker': 'D',
    'alpha': 0.95,
}

fig, ax = plt.subplots(figsize=(11.0 * 0.9, 6.4 * 0.9))
for threshold in DOMAIN_THRESHOLDS:
    meta = domain_threshold_meta_dual[threshold]
    for pool_label in ['Restricted proxies', 'Unrestricted proxies']:
        result = blocking_collateral_results[pool_label][threshold]
        pool_meta = pool_line_meta[pool_label]
        ax.step(
            result['percentages'],
            result['curve'],
            where='post',
            color=meta['color'],
            alpha=pool_meta['alpha'],
            linewidth=2.6,
            linestyle=pool_meta['linestyle'],
            marker=meta['marker'],
            markersize=6.2,
            markerfacecolor=pool_meta['markerfacecolor'],
            markeredgewidth=1.4,
            label=f"{meta['label']} ({pool_meta['suffix']})",
        )

ax.set_ylabel('Top Domains Blocked (%)', fontsize=30)
ax.set_xlabel('Observed Top ASNs Blocked (%)', fontsize=30)
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
style_paper_axes(ax)
format_axes(ax)
ax.tick_params(axis='both', labelsize=20)

annotation_pool = 'Restricted proxies'
top100_curve = blocking_collateral_results[annotation_pool][100]['curve']
line_color = domain_threshold_meta_dual[100]['color']
# These are from trial and error 
annotation_positions = {
    '20440': (4, 78),
    '16509': (12, 38),
    '13335': (42, 62),
    '15169': (67, 68),
}
for annotation in old_figure_annotations[annotation_pool]:
    pct = annotation['pct']
    y_value = top100_curve[pct]
    domains = annotation['domains']
    text_lines = [f"Steep rise: +{annotation['rise']:.1f}%", f"ASN {annotation['asn']} ({len(domains)})"]
    for row in domains:
        text_lines.append(row['domain'])
    label_text = '\n'.join(text_lines)
    x_text, y_text = annotation_positions.get(annotation['asn'], (min(92, pct + 5), min(94, y_value + 12)))
    ax.annotate(
        label_text,
        xy=(pct, y_value),
        xytext=(x_text, y_text),
        textcoords='data',
        fontsize=18,
        color=line_color,
        bbox=dict(facecolor='white', alpha=0.92, edgecolor=line_color, boxstyle='round,pad=0.35'),
        arrowprops=dict(arrowstyle='->', color=line_color, linewidth=1.0),
    )
    ax.scatter([pct], [y_value], color=line_color, s=40, zorder=5)

ax2 = ax.twinx()
combined_frequency_curve = [
    (blocking_coverage_results['Restricted proxies']['frequency']['curve'][pct] + blocking_coverage_results['Unrestricted proxies']['frequency']['curve'][pct]) / 2
    for pct in BLOCKING_PERCENTAGES
]
ax2.step(
    BLOCKING_PERCENTAGES,
    combined_frequency_curve,
    where='post',
    color=proxy_overlay_meta['color'],
    alpha=proxy_overlay_meta['alpha'],
    linewidth=2.8,
    linestyle=proxy_overlay_meta['linestyle'],
    marker=proxy_overlay_meta['marker'],
    markersize=5.8,
    markerfacecolor='white',
    markeredgewidth=1.2,
    label=proxy_overlay_meta['label'],
    zorder=3,
)

ax2.set_ylabel('Observed Snowflakes Blocked (%)', fontsize=22, color=proxy_overlay_meta['color'])
ax2.set_ylim(0, 100)
ax2.tick_params(axis='y', labelsize=20, colors=proxy_overlay_meta['color'], direction='in')
ax2.spines['right'].set_color(proxy_overlay_meta['color'])
ax2.spines['right'].set_linewidth(0.5)
ax2.spines['top'].set_visible(False)
ax2.spines['left'].set_visible(False)
ax2.spines['bottom'].set_visible(False)

handles_left, labels_left = ax.get_legend_handles_labels()
handles_right, labels_right = ax2.get_legend_handles_labels()
fig.legend(handles_left + handles_right, labels_left + labels_right, loc='upper center', bbox_to_anchor=(0.5, 1.10), ncol=2, fontsize=16, frameon=True)
plt.tight_layout(rect=(0, 0, 1, 0.90))
save_paper_figure(fig, 'blocking-collateral-domains-proxy-axis.pdf')
print(FIGURE_OUTPUT_DIR / 'blocking-collateral-domains-proxy-axis.pdf')


## 4. ASN type blocking collateral


In [ ]:
rows_with_ipinfo_by_pool = {
    'Restricted Snowflakes': attach_asn_metadata(restricted_rows, ipinfo_metadata),
    'Unrestricted Snowflakes': attach_asn_metadata(unrestricted_rows, ipinfo_metadata),
}
rows_with_ipinfo_combined = attach_asn_metadata(all_rows, ipinfo_metadata)

def _domain_collateral_by_asn_type(domain_rows, rank_threshold, asn_type):
    filtered_rows = [row for row in domain_rows if row['rank'] <= rank_threshold]
    if not filtered_rows:
        return 0.0
    blocked_count = 0
    for row in filtered_rows:
        row_types = {
            (ipinfo_metadata.get(asn, {}) or {}).get('type', '').strip() or 'unknown'
            for asn in row['asns']
        }
        if asn_type in row_types:
            blocked_count += 1
    return blocked_count / len(filtered_rows) * 100

selected_asn_types = [asn_type for asn_type, count in unique_asns_by_type.most_common() if count >= 5]
pool_type_to_asns = {
    'Restricted Snowflakes': defaultdict(set),
    'Unrestricted Snowflakes': defaultdict(set),
}
for pool_label, rows in rows_with_ipinfo_by_pool.items():
    for row in rows:
        asn = row.get('asn')
        if asn is None:
            continue
        pool_type_to_asns[pool_label][get_asn_type(row)].add(asn)

asn_type_blocking_rows = []
for asn_type in selected_asn_types:
    row = {
        'asn_type': asn_type,
        'restricted_observed_asns': len(pool_type_to_asns['Restricted Snowflakes'].get(asn_type, set())),
        'unrestricted_observed_asns': len(pool_type_to_asns['Unrestricted Snowflakes'].get(asn_type, set())),
        'top100_collateral_pct': _domain_collateral_by_asn_type(top_domain_rows, 100, asn_type),
        'top10k_collateral_pct': _domain_collateral_by_asn_type(top_domain_rows, 10_000, asn_type),
        'top1m_collateral_pct': _domain_collateral_by_asn_type(top_domain_rows, 1_000_000, asn_type),
    }
    for pool_label, rows in rows_with_ipinfo_by_pool.items():
        total_ips = len({r['ip_hash'] for r in rows if r.get('ip_hash')})
        type_ips = {r['ip_hash'] for r in rows if r.get('asn') is not None and get_asn_type(r) == asn_type}
        key = 'restricted_proxy_coverage_pct' if pool_label == 'Restricted Snowflakes' else 'unrestricted_proxy_coverage_pct'
        row[key] = 0.0 if total_ips == 0 else len(type_ips) / total_ips * 100
    asn_type_blocking_rows.append(row)

threshold_marker_meta = {
    'top100_collateral_pct': {'label': '% Top 100 Domains Blocked', 'color': '#1f77b4', 'marker': 'o'},
    'top10k_collateral_pct': {'label': '% Top 10k Domains Blocked', 'color': '#ff7f0e', 'marker': 's'},
    'top1m_collateral_pct': {'label': '% Top 1M Domains Blocked', 'color': '#2ca02c', 'marker': '^'},
}
coverage_bar_meta = {
    'restricted_proxy_coverage_pct': {'label': 'Res. Proxy Coverage %', 'color': '#8b0000', 'alpha': 0.70},
    'unrestricted_proxy_coverage_pct': {'label': 'Unres. Proxy Coverage %', 'color': '#f08080', 'alpha': 0.70},
}
summary_by_type = {
    row['asn_type']: {
        'coverage': max(row['restricted_proxy_coverage_pct'], row['unrestricted_proxy_coverage_pct']),
        'top10k': row['top10k_collateral_pct'],
    }
    for row in asn_type_blocking_rows
}
top_asn_types = [
    asn_type
    for asn_type, _ in sorted(summary_by_type.items(), key=lambda item: (-item[1]['coverage'], item[1]['top10k'], item[0]))[:6]
]
if 'isp' in summary_by_type and 'isp' not in top_asn_types:
    top_asn_types = ['isp'] + [asn_type for asn_type in top_asn_types if asn_type != 'isp'][:5]
row_map = {row['asn_type']: row for row in asn_type_blocking_rows}
positions = list(range(len(top_asn_types)))
offsets = {'restricted_proxy_coverage_pct': -0.18, 'unrestricted_proxy_coverage_pct': 0.18}

fig, ax = plt.subplots(figsize=(10, 5.5))
for coverage_field in ['restricted_proxy_coverage_pct', 'unrestricted_proxy_coverage_pct']:
    meta = coverage_bar_meta[coverage_field]
    ys = [pos + offsets[coverage_field] for pos in positions]
    values = [row_map[asn_type][coverage_field] for asn_type in top_asn_types]
    ax.barh(
        ys,
        values,
        height=0.32,
        color=meta['color'],
        alpha=meta['alpha'],
        edgecolor=meta['color'],
        linewidth=1.0,
        label=meta['label'],
    )
    for y_pos, value in zip(ys, values):
        ax.text(value + 0.8, y_pos, f'{value:.1f}%', va='center', ha='left', fontsize=13, color=meta['color'])
for collateral_field, marker_meta in threshold_marker_meta.items():
    values = [row_map[asn_type][collateral_field] for asn_type in top_asn_types]
    ax.scatter(
        values,
        positions,
        s=95,
        marker=marker_meta['marker'],
        color=marker_meta['color'],
        linewidths=1.2,
        label=marker_meta['label'],
        zorder=4,
    )
for y_pos, asn_type in zip(positions, top_asn_types):
    restricted_count = row_map[asn_type]['restricted_observed_asns']
    unrestricted_count = row_map[asn_type]['unrestricted_observed_asns']
    ax.text(101.5, y_pos + offsets['restricted_proxy_coverage_pct'], f'R:{restricted_count}', va='center', ha='left', fontsize=13, color=coverage_bar_meta['restricted_proxy_coverage_pct']['color'])
    ax.text(101.5, y_pos + offsets['unrestricted_proxy_coverage_pct'], f'U:{unrestricted_count}', va='center', ha='left', fontsize=13, color=coverage_bar_meta['unrestricted_proxy_coverage_pct']['color'])
ax.set_xlabel('Percentage of Observed Proxies Covered (%)', fontsize=25)
ax.set_yticks(positions)
ax.set_yticklabels(top_asn_types, fontsize=24)
ax.invert_yaxis()
ax.set_ylabel('ASN Type', fontsize=25)
ax.set_xlim(0, 110)
style_paper_axes(ax)
format_axes(ax)
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=2, fontsize=22, frameon=True)
plt.tight_layout(rect=(0, 0, 1, 0.90))
format_axes(ax)
ax.tick_params(axis='both', labelsize=22)

save_paper_figure(fig, 'blocking-asn-type-collateral_figure.pdf')
print(FIGURE_OUTPUT_DIR / 'blocking-asn-type-collateral_figure.pdf')

print('ASN-type blocking effect (combined collateral, split proxy coverage):')
for row in sorted(asn_type_blocking_rows, key=lambda item: (-max(item['restricted_proxy_coverage_pct'], item['unrestricted_proxy_coverage_pct']), item['asn_type'])):
    print(f"  {row['asn_type']:<12} restricted={row['restricted_proxy_coverage_pct']:.1f}% | unrestricted={row['unrestricted_proxy_coverage_pct']:.1f}% | top100={row['top100_collateral_pct']:.1f}% | top10k={row['top10k_collateral_pct']:.1f}% | top1M={row['top1m_collateral_pct']:.1f}% | R ASNs={row['restricted_observed_asns']:,} | U ASNs={row['unrestricted_observed_asns']:,}")


## 5. Retries when previously seen entities are blocked


In [ ]:
def _sorted_rows_for_retry(rows):
    return sorted(rows, key=lambda row: row['raw']['Date'])


def _average_retries_if_blocked(rows, field, ignore_missing=True):
    ordered_rows = _sorted_rows_for_retry(rows)
    grouped = defaultdict(list)
    for row in ordered_rows:
        grouped[row['date']].append(row)

    seen_before = set()
    metrics = []
    for day in sorted(grouped):
        day_rows = grouped[day]
        day_values = []
        for row in day_rows:
            value = row.get(field)
            if value is None and ignore_missing:
                continue
            cleaned = str(value).strip() if value is not None else ''
            if not cleaned and ignore_missing:
                continue
            day_values.append(cleaned)
        if not day_values:
            continue
        novel_count = sum(1 for value in day_values if value not in seen_before)
        total_count = len(day_values)
        p_success = novel_count / total_count
        average_retries = None if p_success == 0 else (1 - p_success) / p_success
        metrics.append({
            'date': day,
            'novel_count': novel_count,
            'total_count': total_count,
            'p_success': p_success,
            'average_retries': average_retries,
        })
        seen_before |= set(day_values)
    return metrics


retry_curve_results = {
    'Restricted proxies': {
        'ip_hash': _average_retries_if_blocked(restricted_rows, 'ip_hash', ignore_missing=False),
        'asn': _average_retries_if_blocked(restricted_rows, 'asn', ignore_missing=True),
    },
    'Unrestricted proxies': {
        'ip_hash': _average_retries_if_blocked(unrestricted_rows, 'ip_hash', ignore_missing=False),
        'asn': _average_retries_if_blocked(unrestricted_rows, 'asn', ignore_missing=True),
    },
}

retry_line_meta = {
    ('Restricted proxies', 'ip_hash'): {'label': 'Previously Seen Proxies Blocked (R)', 'color': '#8b0000', 'linestyle': '-', 'marker': 'o'},
    ('Restricted proxies', 'asn'): {'label': 'Previously Seen ASNs Blocked (R)', 'color': '#0b4f8a', 'linestyle': '--', 'marker': 's'},
    ('Unrestricted proxies', 'ip_hash'): {'label': 'Previously Seen Proxies Blocked (U)', 'color': '#f08080', 'linestyle': '-', 'marker': '^'},
    ('Unrestricted proxies', 'asn'): {'label': 'Previously Seen ASNs Blocked (U)', 'color': '#7fb3d5', 'linestyle': '--', 'marker': 'D'},
}

fig, ax = plt.subplots(figsize=(9.5, 5.5))
for pool_label in ['Restricted proxies', 'Unrestricted proxies']:
    for entity_key in ['ip_hash', 'asn']:
        metrics = retry_curve_results[pool_label][entity_key]
        dates = [entry['date'] for entry in metrics if entry['average_retries'] is not None]
        retries = [entry['average_retries'] for entry in metrics if entry['average_retries'] is not None]
        meta = retry_line_meta[(pool_label, entity_key)]
        ax.plot(
            dates,
            retries,
            color=meta['color'],
            linestyle=meta['linestyle'],
            linewidth=2.4,
            marker=meta['marker'],
            markersize=5.8,
            label=meta['label'],
        )
ax.set_ylabel('Average # Retries Before Success', fontsize=24)
ax.set_xlabel('Date', fontsize=24)
ax.set_yscale('log')
style_paper_axes(ax)
format_axes(ax)
ax.tick_params(axis='x', rotation=35)
ax.tick_params(axis='both', labelsize=22)
for window in IRAN_CONTEXT_WINDOWS:
    ax.axvspan(window['start'], window['end'], color=window['color'], alpha=window['alpha'])
handles, labels = ax.get_legend_handles_labels()
handles.extend([
    Patch(facecolor=IRAN_CONTEXT_WINDOWS[0]['color'], alpha=IRAN_CONTEXT_WINDOWS[0]['alpha'], label=IRAN_CONTEXT_WINDOWS[0]['label']),
    Patch(facecolor=IRAN_CONTEXT_WINDOWS[1]['color'], alpha=IRAN_CONTEXT_WINDOWS[1]['alpha'], label=IRAN_CONTEXT_WINDOWS[1]['label']),
    Patch(facecolor=IRAN_CONTEXT_WINDOWS[2]['color'], alpha=IRAN_CONTEXT_WINDOWS[2]['alpha'], label=IRAN_CONTEXT_WINDOWS[2]['label']),
])
labels.extend([window['label'] for window in IRAN_CONTEXT_WINDOWS])
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=2, fontsize=18, frameon=True)
plt.tight_layout(rect=(0, 0, 1, 0.88))
format_axes(ax)
ax.tick_params(axis='both', labelsize=22)

save_paper_figure(fig, 'blocking-retries-previously-seen_figure.pdf')
print(FIGURE_OUTPUT_DIR / 'blocking-retries-previously-seen_figure.pdf')
